In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.neighbors import KNeighborsRegressor

RANDOM_STATE = 42

# Load Model & Preprocessing

In [32]:
file_path = 'data/ae_only_unambiguous_1000.csv'
df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='ignore', dtype={'lang5.x': str, 'lang6.x': str})

In [33]:
df_target = df.groupby('website')['response.x'].mean().reset_index()

df_target = df_target.rename(columns={
    'website': 'image_name',
    'response.x': 'skor_estetika'
})

df_target['image_name'] = df_target['image_name'] + '.png'
df_target.head(5)

,image_name,skor_estetika
0,english_0.png,2.290287
1,english_1.png,4.195294
2,english_10.png,3.926130
3,english_100.png,4.958242
4,english_101.png,4.864776


In [34]:
df_target.shape

(418, 2)

In [35]:
df_cnn = pd.read_csv('data/ekstrak_cnn.csv')
df_cv = pd.read_csv('data/ekstrak_matematika_opencv.csv')

df_hybrid = pd.merge(df_cnn, df_cv, on='image_name', how='inner')
df_hybrid.head(5)

,image_name,cnn_feat_0,cnn_feat_1,cnn_feat_2,cnn_feat_3,cnn_feat_4,cnn_feat_5,cnn_feat_6,cnn_feat_7,cnn_feat_8,...,purple,fuchsia,green,lime,olive,yellow,navy,blue,teal,aqua
0,english_0.png,-4.721025,2.358559,-3.684831,0.463756,-0.524554,1.350653,-0.075894,1.700438,0.542055,...,0.000000,0.000000,0.000544,0.000000,0.000547,0.000252,0.019568,0.000340,0.005155,0.001189
1,english_1.png,1.210117,5.665122,-3.581102,-1.811846,-4.932022,0.047140,-0.264386,1.231203,0.168720,...,0.000048,0.000160,0.000078,0.000011,0.002449,0.002518,0.001640,0.000699,0.075212,0.003403
2,english_10.png,-2.391359,3.296005,0.787008,0.606368,0.721851,0.919729,0.678484,-0.327567,-0.328909,...,0.007382,0.000051,0.000003,0.000000,0.001636,0.001023,0.009420,0.000594,0.015256,0.006435
3,english_100.png,-0.749734,0.351490,4.590096,2.475029,-1.191490,-3.183770,3.293322,-1.044738,0.705354,...,0.000027,0.000000,0.000000,0.000000,0.085815,0.054267,0.000006,0.000000,0.005960,0.000017
4,english_101.png,4.423751,-1.595877,-1.300480,1.046409,0.702248,3.977708,-0.080589,-2.913515,1.900459,...,0.000015,0.000000,0.021142,0.000000,0.008461,0.000000,0.153999,0.000011,0.057641,0.000156


In [36]:
df_hybrid.shape

(410, 53)

In [37]:
df_final = pd.merge(df_hybrid, df_target, on='image_name', how='inner')
df_final.head(5)

,image_name,cnn_feat_0,cnn_feat_1,cnn_feat_2,cnn_feat_3,cnn_feat_4,cnn_feat_5,cnn_feat_6,cnn_feat_7,cnn_feat_8,...,fuchsia,green,lime,olive,yellow,navy,blue,teal,aqua,skor_estetika
0,english_0.png,-4.721025,2.358559,-3.684831,0.463756,-0.524554,1.350653,-0.075894,1.700438,0.542055,...,0.000000,0.000544,0.000000,0.000547,0.000252,0.019568,0.000340,0.005155,0.001189,2.290287
1,english_1.png,1.210117,5.665122,-3.581102,-1.811846,-4.932022,0.047140,-0.264386,1.231203,0.168720,...,0.000160,0.000078,0.000011,0.002449,0.002518,0.001640,0.000699,0.075212,0.003403,4.195294
2,english_10.png,-2.391359,3.296005,0.787008,0.606368,0.721851,0.919729,0.678484,-0.327567,-0.328909,...,0.000051,0.000003,0.000000,0.001636,0.001023,0.009420,0.000594,0.015256,0.006435,3.926130
3,english_100.png,-0.749734,0.351490,4.590096,2.475029,-1.191490,-3.183770,3.293322,-1.044738,0.705354,...,0.000000,0.000000,0.000000,0.085815,0.054267,0.000006,0.000000,0.005960,0.000017,4.958242
4,english_101.png,4.423751,-1.595877,-1.300480,1.046409,0.702248,3.977708,-0.080589,-2.913515,1.900459,...,0.000000,0.021142,0.000000,0.008461,0.000000,0.153999,0.000011,0.057641,0.000156,4.864776


In [38]:
df_final.shape

(398, 54)

In [39]:
X_cv = pd.merge(df_cv, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cv.shape

(398, 22)

In [40]:
X_cnn = pd.merge(df_cnn, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cnn.shape

(398, 30)

In [41]:
X = df_final.drop(columns=['image_name', 'skor_estetika'])
y = df_final['skor_estetika']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Building Model

## Linear Regression

### 1.1 Training dan Prediksi Dasar

In [42]:
lr_model = LinearRegression()

t0 = time.perf_counter()
lr_model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = lr_model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, lr_model.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score (test)  : {r2_test:.4f}")
print(f"R2 Score (train) : {r2_train:.4f}")
print(f"RMSE             : {rmse:.4f}")
print(f"Waktu training   : {waktu_train:.4f} detik")
print(f"Waktu prediksi   : {waktu_predict:.4f} detik")


R2 Score (test)  : 0.2715
R2 Score (train) : 0.6507
RMSE             : 0.7861
Waktu training   : 0.0055 detik
Waktu prediksi   : 0.0008 detik


### 1.2 Model Fine-Tuning

In [43]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Parameter grid untuk Linear Regression
param_grid = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

# Grid search dengan 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=LinearRegression(),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

best_lr = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

r2_test = best_lr.predict(X_test_scaled)
r2_train = best_lr.predict(X_train_scaled)
cv_r2 = grid_search.best_score_

print(f"Tuned R2  (test) : {r2_score(y_test, r2_test):.4f}")
print(f"Tuned R2 (train) : {r2_score(y_train, r2_train):.4f}")
print(f"Mean CV R2 (Real Train/Val Baseline): {cv_r2:.4f}")
print(f"Tuned RMSE       : {np.sqrt(mean_squared_error(y_test, r2_test)):.4f}")

Best Parameters: {'fit_intercept': True, 'positive': False}
Tuned R2  (test) : 0.2715
Tuned R2 (train) : 0.6507
Mean CV R2 (Real Train/Val Baseline): -0.0070
Tuned RMSE       : 0.7861


### 1.3 Training Best-Model


In [44]:
lr_model = LinearRegression(fit_intercept=True, positive=False)

t0 = time.perf_counter()
lr_model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = lr_model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred)
r2_train = r2_score(y_train, lr_model.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score (test)  : {r2_test:.4f}")
print(f"R2 Score (train) : {r2_train:.4f}")
print(f"RMSE             : {rmse:.4f}")
print(f"Waktu training   : {waktu_train:.4f} detik")
print(f"Waktu prediksi   : {waktu_predict:.4f} detik")

R2 Score (test)  : 0.2715
R2 Score (train) : 0.6507
RMSE             : 0.7861
Waktu training   : 0.0070 detik
Waktu prediksi   : 0.0007 detik


## K-Nearest-Neighbours

### 1.1 Training dan Prediksi Dasar

In [45]:
knn = KNeighborsRegressor(n_neighbors=5, weights='uniform')

t0 = time.perf_counter()
knn.fit(X_train_scaled, y_train)
waktu_train_knn = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_scaled)
waktu_predict_knn = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred_knn)
r2_train = r2_score(y_train, knn.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print(f"R2 Score (test)    : {r2_test:.4f}")
print(f"R2 Score (train)   : {r2_train:.4f}")
print(f"RMSE               : {rmse:.4f}")
print(f"Waktu training     : {waktu_train_knn:.4f} detik")
print(f"Waktu prediksi     : {waktu_predict_knn:.4f} detik")

R2 Score (test)    : 0.2579
R2 Score (train)   : 0.4423
RMSE               : 0.7934
Waktu training     : 0.0018 detik
Waktu prediksi     : 0.0099 detik


### 1.2 Model Fine-Tuning

In [46]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [7, 8, 9, 10, 11, 12, 13, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'cosine']
}

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=KNeighborsRegressor(),
    param_grid=param_grid,
    scoring='r2',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

best_knn = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)

r2_test = best_knn.predict(X_test_scaled)
r2_train = best_knn.predict(X_train_scaled)
cv_r2 = grid_search.best_score_

print(f"Tuned R2  (test) : {r2_score(y_test, r2_test):.4f}")
print(f"Tuned R2 (train) : {r2_score(y_train, r2_train):.4f}")
print(f"Mean CV R2 (Real Train/Val Baseline): {cv_r2:.4f}")
print(f"Tuned RMSE       : {np.sqrt(mean_squared_error(y_test, r2_test)):.4f}")

Best Parameters: {'metric': 'cosine', 'n_neighbors': 24, 'weights': 'distance'}
Tuned R2  (test) : 0.3745
Tuned R2 (train) : 1.0000
Mean CV R2 (Real Train/Val Baseline): 0.3272
Tuned RMSE       : 0.7284


### 1.3 Training Best Model

In [47]:
knn = KNeighborsRegressor(n_neighbors=24, weights='distance', metric='cosine')

t0 = time.perf_counter()
knn.fit(X_train_scaled, y_train)
waktu_train_knn = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_scaled)
waktu_predict_knn = time.perf_counter() - t0

r2_test = r2_score(y_test, y_pred_knn)
r2_train = r2_score(y_train, knn.predict(X_train_scaled))
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print(f"R2 Score (test)    : {r2_test:.4f}")
print(f"R2 Score (train)   : {r2_train:.4f}")
print(f"RMSE               : {rmse:.4f}")
print(f"Waktu training     : {waktu_train_knn:.4f} detik")
print(f"Waktu prediksi     : {waktu_predict_knn:.4f} detik")

R2 Score (test)    : 0.3745
R2 Score (train)   : 1.0000
RMSE               : 0.7284
Waktu training     : 0.0011 detik
Waktu prediksi     : 0.0020 detik


## Decision Tree

In [48]:
from sklearn.tree import DecisionTreeRegressor
dt_model = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=RANDOM_STATE)

t0 = time.perf_counter()
dt_model.fit(X_train_scaled, y_train)
waktu_train_dt = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_dt = dt_model.predict(X_test_scaled)
waktu_predict_dt = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred_dt)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print(f"R2 Score (test)     : {r2:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"Waktu training  : {waktu_train_dt:.4f} detik")
print(f"Waktu prediksi  : {waktu_predict_dt:.4f} detik")

r2_train = r2_score(y_train, dt_model.predict(X_train_scaled))
print(f"R2 Score (train) : {r2_train:.4f}")

R2 Score (test)     : -0.0069
RMSE            : 0.9242
Waktu training  : 0.0081 detik
Waktu prediksi  : 0.0004 detik
R2 Score (train) : 0.4554


# Evaluation

# Model Overfitting

In [49]:
kolom_cnn = [col for col in df_hybrid.columns if 'cnn_feat' in col]
kolom_cv = [col for col in df_hybrid.columns if col not in kolom_cnn and col not in ['image_name', 'skor_estetika']]

matriks_korelasi = df_hybrid[kolom_cnn + kolom_cv].corr()

for i in range(30):
  target_fitur = f'cnn_feat_{str(i)}'
  korelasi_feat_7 = matriks_korelasi.loc[target_fitur, kolom_cv]
  korelasi_feat_7_sorted = korelasi_feat_7.abs().sort_values(ascending=False)

  print("\n======================================================")
  print(f"Korelasi tertinggi untuk {target_fitur} dengan fitur OpenCV:")
  print(korelasi_feat_7.loc[korelasi_feat_7_sorted.index].head())


Korelasi tertinggi untuk cnn_feat_0 dengan fitur OpenCV:
white              -0.535615
quadtree_leaves     0.517448
image_area_ratio    0.411305
colorfulness        0.335544
gray                0.291772
Name: cnn_feat_0, dtype: float64

Korelasi tertinggi untuk cnn_feat_1 dengan fitur OpenCV:
white               0.482866
black              -0.430684
symmetry           -0.345834
maroon             -0.304175
image_area_ratio    0.217094
Name: cnn_feat_1, dtype: float64

Korelasi tertinggi untuk cnn_feat_2 dengan fitur OpenCV:
black     -0.365272
white      0.340232
teal      -0.233975
balance    0.201654
yellow     0.170520
Name: cnn_feat_2, dtype: float64

Korelasi tertinggi untuk cnn_feat_3 dengan fitur OpenCV:
quadtree_leaves    -0.532910
colorfulness       -0.386566
red                -0.323567
image_area_ratio   -0.225984
maroon             -0.208791
Name: cnn_feat_3, dtype: float64

Korelasi tertinggi untuk cnn_feat_4 dengan fitur OpenCV:
black               0.298145
quadtree_leave

# Result Analysis

# Model Export

In [50]:
import joblib
joblib.dump(scaler, 'models/hybrid_scaler.pkl')
joblib.dump(knn, 'models/ml_model.pkl')

['models/ml_model.pkl']